In [11]:
# Importación de librerías y módulos necesarios

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from imblearn.under_sampling import RandomUnderSampler

In [12]:
# Carga y muestreo optimizado (según curva de aprendizaje)

df_train_full = pd.read_csv('df_voices_train.csv')
df_eval = pd.read_csv('df_voices_eval.csv')

# Seleccionamos las 6000 muestras según el punto de estabilidad encontrado

df_train = df_train_full.sample(n=6000, random_state=42).reset_index(drop=True)

In [13]:
# Preparación de datos

def prepare_data_for_testing(df):
    df['y'] = df['Key'].map({'bonafide': 0, 'spoof': 1})
    drop_cols = ['Key', 'file_name', 'User_ID', 'Spoofing_ID', 'y', 'Gender']
    X = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')
    y = df['y']
    s_ids = df['Spoofing_ID']
    gender = df['Gender'] if 'Gender' in df.columns else None
    return X, y, s_ids, gender

X_train, y_train, _, _ = prepare_data_for_testing(df_train)
X_eval, y_eval, spoof_ids_eval, gender_eval = prepare_data_for_testing(df_eval)

In [14]:
# Entrenamiento con parámetros ganadores

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_eval_scaled = scaler.transform(X_eval)

model_final = RandomForestClassifier(
    n_estimators=250,
    random_state=42,
    n_jobs=-1
)
model_final.fit(X_train_scaled, y_train)

def get_metrics(y_true, y_pred):
    return {
        'F1': f1_score(y_true, y_pred),
        'Prec': precision_score(y_true, y_pred, zero_division=0),
        'Rec': recall_score(y_true, y_pred)
    }

print(f"Modelo entrenado con {len(df_train)} muestras.")

Modelo entrenado con 6000 muestras.


In [15]:
# Test 1: Rendimiento por tipo de ataque

print("\nTest 1: Rendimiento por tipo de ataque")
resultados_ataque = []
for ataque in spoof_ids_eval.unique():
    mask = (spoof_ids_eval == ataque)
    if mask.sum() > 0:
        y_p = model_final.predict(X_eval_scaled[mask])

        # Para ataques individuales, el Recall es más descriptivo (cuántos detectó)
        m = get_metrics(y_eval[mask], y_p)
        resultados_ataque.append({'ID_Ataque': ataque, 'F1-Score': m['F1'], 'Recall': m['Rec']})

df_res_ataque = pd.DataFrame(resultados_ataque).sort_values(by='F1-Score', ascending=False)
print(df_res_ataque.to_string(index=False))


Test 1: Rendimiento por tipo de ataque
ID_Ataque  F1-Score   Recall
      A10  1.000000 1.000000
      A11  1.000000 1.000000
      A14  1.000000 1.000000
      A09  1.000000 1.000000
      A08  1.000000 1.000000
      A13  1.000000 1.000000
      A07  1.000000 1.000000
      A15  0.999695 0.999389
      A12  0.999695 0.999389
      A16  0.997552 0.995116
      A18  0.997450 0.994912
      A19  0.944701 0.895197
      A17  0.929995 0.869149
        -  0.000000 0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [16]:
# Test 2: Sesgo por género

print("\nTest 2: Sesgo por género")
for g in gender_eval.unique():
    mask = (gender_eval == g)
    y_p = model_final.predict(X_eval_scaled[mask])
    m = get_metrics(y_eval[mask], y_p)
    print(f"Género {g:10} | F1: {m['F1']:.4f} | Prec: {m['Prec']:.4f} | Rec: {m['Rec']:.4f}")


Test 2: Sesgo por género
Género male       | F1: 0.9641 | Prec: 0.9417 | Rec: 0.9876
Género female     | F1: 0.9557 | Prec: 0.9343 | Rec: 0.9781


In [17]:
# Test 3: Resistencia al ruido

print("\nTest 3: Resistencia al ruido (0.05)")
X_eval_noisy = X_eval_scaled + 0.05 * np.random.normal(size=X_eval_scaled.shape)
y_p_noisy = model_final.predict(X_eval_noisy)
m_noisy = get_metrics(y_eval, y_p_noisy)
m_orig = get_metrics(y_eval, model_final.predict(X_eval_scaled))
print(f"F1 Original: {m_orig['F1']:.4f} -> F1 Ruidoso: {m_noisy['F1']:.4f}")
print(f"Caída F1: {m_orig['F1'] - m_noisy['F1']:.4f}")


Test 3: Resistencia al ruido (0.05)
F1 Original: 0.9583 -> F1 Ruidoso: 0.9580
Caída F1: 0.0003


In [18]:
# Test 4: Degradación telefónica (simulada)

print("\nTest 4: Degradación telefónica (simulada)")
X_eval_tel = X_eval_scaled.copy()
cols_brillo = [i for i, col in enumerate(X_train.columns) if 'rolloff' in col or 'centroid' in col]
X_eval_tel[:, cols_brillo] = 0 # Eliminamos info de frecuencias altas
y_p_tel = model_final.predict(X_eval_tel)
m_tel = get_metrics(y_eval, y_p_tel)
print(f"F1 tras pérdida de frecuencias: {m_tel['F1']:.4f}")


Test 4: Degradación telefónica (simulada)
F1 tras pérdida de frecuencias: 0.9584


In [19]:
# Test 5: Zona gris

probs = model_final.predict_proba(X_eval_scaled)[:, 1]
incertidumbre = (probs > 0.40) & (probs < 0.60) # Ampliamos un poco el rango de duda
df_duda = df_eval[incertidumbre]
print(f"\nTest 5: Zona gris")
print(f"Audios con baja certeza: {len(df_duda)}")
if len(df_duda) > 0:
    print("Top ataques en zona de duda:")
    print(df_duda['Spoofing_ID'].value_counts().head(3))


Test 5: Zona gris
Audios con baja certeza: 3558
Top ataques en zona de duda:
Spoofing_ID
-      1408
A17     949
A19     939
Name: count, dtype: int64
